In [ ]:
# Clase sesión 2 del módulo 9, 31/08
#  0) Instalar PySpark e iniciar la sesión
!pip -q install pyspark
# ! es un “magic” de Colab/Jupyter para ejecutar comandos de consola.
# pip install pyspark instala PySpark en el entorno actual.
# -q = quiet (silencioso) para no imprimir todo el log de instalación.
from pyspark.sql import SparkSession
# Importa SparkSession, la puerta de entrada “todo en uno” a Spark (reemplaza a SQLContext y HiveContext y expone el SparkContext como spark.sparkContext).

spark = SparkSession.builder \
    .appName("AnalisisPenguins") \
    .master("local[*]") \
    .getOrCreate()
# SparkSession.builder crea un constructor de sesión.
# .appName("AnalisisPenguins") pone el nombre de tu aplicación (aparece en la UI/logs).
# .master("local[*]") indica dónde correr Spark:
# local[*] = modo local usando todos los núcleos disponibles.
# (Notas: local[1] fuerza 1 hilo; en clúster sería yarn, mesos, o una URL de standalone.)
# .getOrCreate() crea la sesión si no existe, o reuse una ya creada (evita errores por sesiones duplicadas).

print("✅ Spark listo:", spark.version)

✅ Spark listo: 3.5.1


In [ ]:
# 1) Descargar dataset público (Penguins) y cargarlo con Spark
# =========================================
# 1) DESCARGA DEL CSV DESDE INTERNET
# =========================================
# Fuente confiable (repositorio de seaborn en GitHub)
!wget -q -O /content/penguins.csv https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv
# !wget → ejecuta un comando de terminal desde Colab/Jupyter (el ! manda a la shell).
# -q (quiet) → descarga en modo silencioso (menos texto en la consola).
# -O /content/penguins.csv → guarda el archivo con ese nombre y ruta.
# En Colab, /content es tu carpeta de trabajo.
# La URL es el CSV público del dataset penguins (repo oficial de seaborn en GitHub).
# Tip: puedes comprobar que quedó guardado con !ls -lh /content/penguins.csv o ver primeras líneas con !head /content/penguins.csv.

# Cargar con Spark
df = spark.read.option("header", True).option("inferSchema", True).csv("/content/penguins.csv")
# spark.read.csv(...) → lee un CSV y crea un DataFrame de Spark (tabla distribuida).
# .option("header", True) → le dice a Spark que la primera fila del CSV tiene nombres de columnas.
# .option("inferSchema", True) → Spark detecta tipos (int, double, string, …) automáticamente.
# Útil para clases/demos; en producción suele ser mejor definir el esquema explícitamente para evitar errores de inferencia y acelerar la lectura.
print("Filas:", df.count())
# count() es una acción: obliga a Spark a leer el archivo y contar las filas (dispara la ejecución “lazy”).
# Imprime cuántos registros cargaste.
df.printSchema()
# Muestra el esquema: nombre de cada columna, tipo de dato inferido y si puede ser NULL.
# Es ideal para verificar que Spark interpretó bien las columnas (por ejemplo, que números no quedaron como string).
df.show(5, truncate=False)
# Enseña las primeras 5 filas.
# truncate=False evita que Spark corte las cadenas largas; ves los valores completos.

Filas: 344
root
 |-- species: string (nullable = true)
 |-- island: string (nullable = true)
 |-- bill_length_mm: double (nullable = true)
 |-- bill_depth_mm: double (nullable = true)
 |-- flipper_length_mm: integer (nullable = true)
 |-- body_mass_g: integer (nullable = true)
 |-- sex: string (nullable = true)

+-------+---------+--------------+-------------+-----------------+-----------+------+
|species|island   |bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|sex   |
+-------+---------+--------------+-------------+-----------------+-----------+------+
|Adelie |Torgersen|39.1          |18.7         |181              |3750       |MALE  |
|Adelie |Torgersen|39.5          |17.4         |186              |3800       |FEMALE|
|Adelie |Torgersen|40.3          |18.0         |195              |3250       |FEMALE|
|Adelie |Torgersen|NULL          |NULL         |NULL             |NULL       |NULL  |
|Adelie |Torgersen|36.7          |19.3         |193              |3450       |FEMALE

Variable	Descripción	Unidad	Tipo estadístico	Tipo en Spark
1. species	Especie del pingüino (p. ej., Adelie, Gentoo, Chinstrap)	—	Cualitativa nominal (categoría)	string
2. island	Isla de muestreo (p. ej., Torgersen, Dream, Biscoe)	—	Cualitativa nominal (categoría)	string
3. bill_length_mm	Longitud del pico	milímetros	Cuantitativa continua	double
4. bill_depth_mm	Profundidad (alto) del pico	milímetros	Cuantitativa continua	double
5. flipper_length_mm	Longitud de la aleta	milímetros	Cuantitativa continua (registrada como entero)	integer
6. body_mass_g	Masa corporal	gramos	Cuantitativa continua (registrada como entero)	integer
7. sex	Sexo del individuo (p. ej., MALE, FEMALE)	—	Cualitativa nominal (categoría)	string

lectura de la salida, de arriba a abajo:

Filas: 344

Es el resultado de df.count(). Indica que tu DataFrame tiene 344 registros (pingüinos).

count() es una acción en Spark (dispara la lectura real del CSV).

Esquema (printSchema())

root
 |-- species: string (nullable = true)
 |-- island: string (nullable = true)
 |-- bill_length_mm: double (nullable = true)
 |-- bill_depth_mm: double (nullable = true)
 |-- flipper_length_mm: integer (nullable = true)
 |-- body_mass_g: integer (nullable = true)
 |-- sex: string (nullable = true)


Cada línea es una columna con su tipo Spark:

string: texto (p. ej., Adelie, Torgersen, MALE).

double: numérico con decimales (longitudes de pico en mm).

integer: numérico entero (aletas en mm, masa en gramos).

nullable = true significa que puede tener NULL (faltantes). De hecho, luego ves un renglón con varios NULL.

df.show(5, truncate=False)
Muestra las primeras 5 filas completas, sin truncar texto:

|Adelie |Torgersen|39.1|18.7|181|3750|MALE  |
|Adelie |Torgersen|39.5|17.4|186|3800|FEMALE|
|Adelie |Torgersen|40.3|18.0|195|3250|FEMALE|
|Adelie |Torgersen|NULL|NULL|NULL|NULL|NULL |
|Adelie |Torgersen|36.7|19.3|193|3450|FEMALE|


Interpretación rápida de columnas:

species (especie), island (isla).

bill_length_mm / bill_depth_mm: largo/profundidad del pico (mm).

flipper_length_mm: longitud de aleta (mm) — entero.

body_mass_g: masa corporal (gramos) — entero.

sex: sexo.

La fila 4 tiene valores NULL en las medidas (faltantes). Es normal en datos reales.

¿Qué haría a continuación?

Contar nulos por columna y decidir si eliminas esas filas o imputas.

Ver estadísticas para chequear rangos y unidades.

In [ ]:
# 2) Limpieza rápida (nulos) + exploración
# =========================================
# 2) LIMPIEZA Y EXPLORACIÓN
# =========================================
from pyspark.sql import functions as F
# Trae utilidades para manipular columnas: F.col, F.when, F.avg, F.round, etc.
# Usamos el alias F para escribir más corto

# Conteo de nulos por columna
print("=== Nulos por columna ===")
for c in df.columns:
    print(f"{c:22s} ->", df.filter(F.col(c).isNull()).count())
# Recorre todas las columnas de df.
# F.col(c).isNull() crea una condición “la columna c es NULL”.
# df.filter(...).count() aplica el filtro y cuenta cuántas filas cumplen → dispara ejecución (acción).
# Así ves dónde hay valores faltantes antes de limpiar.
# Más eficiente (una sola pasada)

# Para facilitar análisis numéricos, eliminamos nulos en las columnas clave
cols_clave = ["species","island","sex","bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
df_clean = df.dropna(subset=cols_clave)
# subset=cols_clave indica que solo miramos nulos en esas columnas.
# dropna con how="any" (por defecto) elimina toda fila que tenga al menos un NULL en ese subconjunto.
# Resultado: df_clean tiene registros completos para el análisis básico.
# Cuidado: si hay muchos nulos, puedes perder demasiadas filas.
# Alternativas:
# Imputar numéricas (mediana/media) con F.when(...).otherwise(...) o Imputer de MLlib.
# Rellenar categóricas: df.fillna({"sex": "UNKNOWN"}).
# Limpiar solo las columnas que vas a usar en cada análisis.

# Ver cuántas filas quedaron y una muestra
print("\nFilas después de limpieza:", df_clean.count())
df_clean.select(cols_clave).show(5, truncate=False)
# count() confirma el tamaño final (acción).
# select(...).show(...) enseña 5 filas completas de las columnas clave.

# Descripción numérica básica
df_clean.select("bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g").describe().show()
# Describe() devuelve estadísticos descriptivos: count, mean, stddev, min, max.
# Como pasaste solo numéricas, todas esas métricas tienen sentido (para string, mean/stddev salen nulos).
# El resultado es otro DataFrame (con una columna summary + columnas de tus métricas).

=== Nulos por columna ===
species                -> 0
island                 -> 0
bill_length_mm         -> 2
bill_depth_mm          -> 2
flipper_length_mm      -> 2
body_mass_g            -> 2
sex                    -> 11

Filas después de limpieza: 333
+-------+---------+------+--------------+-------------+-----------------+-----------+
|species|island   |sex   |bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|
+-------+---------+------+--------------+-------------+-----------------+-----------+
|Adelie |Torgersen|MALE  |39.1          |18.7         |181              |3750       |
|Adelie |Torgersen|FEMALE|39.5          |17.4         |186              |3800       |
|Adelie |Torgersen|FEMALE|40.3          |18.0         |195              |3250       |
|Adelie |Torgersen|FEMALE|36.7          |19.3         |193              |3450       |
|Adelie |Torgersen|MALE  |39.3          |20.6         |190              |3650       |
+-------+---------+------+--------------+-------------+--

1) Nulos por columna
=== Nulos por columna ===
species                -> 0
island                 -> 0
bill_length_mm         -> 2
bill_depth_mm          -> 2
flipper_length_mm      -> 2
body_mass_g            -> 2
sex                    -> 11


Interpretación:

No faltan datos en species ni island.

En las cuatro numéricas hay 2 nulos cada una.

sex tiene 11 nulos.

Conclusión: hay 11 filas que tienen al menos un nulo en las columnas clave. Lo confirma el conteo tras limpiar (344 → 333). Eso implica que los 2 nulos numéricos probablemente coinciden con filas que ya estaban nulas en sex (no son 2+2+2+2+11 filas distintas, sino que el conjunto unión de filas con nulos es 11).

2) Filas después de limpieza + muestra
Filas después de limpieza: 333
+-------+---------+------+--------------+-------------+-----------------+-----------+
|species|island   |sex   |bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|
+-------+---------+------+--------------+-------------+-----------------+-----------+
|Adelie |Torgersen|MALE  |39.1          |18.7         |181              |3750       |
|Adelie |Torgersen|FEMALE|39.5          |17.4         |186              |3800       |
|Adelie |Torgersen|FEMALE|40.3          |18.0         |195              |3250       |
|Adelie |Torgersen|FEMALE|36.7          |19.3         |193              |3450       |
|Adelie |Torgersen|MALE  |39.3          |20.6         |190              |3650       |


Interpretación:

Partías con 344 registros, tras dropna(subset=...) quedan 333 (se eliminaron 11 filas con nulos en las columnas clave).

La muestra (top 5) muestra valores completos y coherentes en mm (pico/alet a) y g (masa), y sexo categórico.

3) Descripción numérica básica (sobre 333 filas limpias)
+-------+------------------+------------------+------------------+-----------------+
|summary|    bill_length_mm|     bill_depth_mm| flipper_length_mm|      body_mass_g|
+-------+------------------+------------------+------------------+-----------------+
|  count|               333|               333|               333|              333|
|   mean|43.992792792792805| 17.16486486486487|200.96696696696696|4207.057057057057|
| stddev| 5.468668342647557|1.9692354633199007|14.015765288287918|805.2158019428971|
|    min|              32.1|              13.1|               172|             2700|
|    max|              59.6|              21.5|               231|             6300|


count: 333 observaciones válidas en cada variable numérica.

mean (media):

Longitud de pico ≈ 44.0 mm

Profundidad de pico ≈ 17.16 mm

Longitud de aleta ≈ 200.97 mm

Masa ≈ 4207 g

stddev (desviación estándar muestral):

Cuánta variabilidad hay alrededor de la media (p. ej., masa ± 805 g).

min / max:

Rangos observados: p. ej., aleta entre 172–231 mm, masa entre 2700–6300 g.

Lectura didáctica:

Estos valores agregan todas las especies; si comparas especies (Adelie, Gentoo, Chinstrap), verás medias distintas.
Ejercicio sugerido: agrupa por species y calcula medias:

from pyspark.sql import functions as F
df_clean.groupBy("species").agg(
    F.round(F.avg("bill_length_mm"),2).alias("mean_bill_len"),
    F.round(F.avg("bill_depth_mm"),2).alias("mean_bill_dep"),
    F.round(F.avg("flipper_length_mm"),2).alias("mean_flipper"),
    F.round(F.avg("body_mass_g"),2).alias("mean_mass")
).show()

En resumen

Detectaste y limpiaste nulos correctamente (quedaron 333 filas completas).

Las medidas centrales y rangos son consistentes con datos biométricos de pingüinos.

Para análisis más finos, conviene estratificar por species o sex y comparar estadísticas entre grupos (ANOVA, boxplots por grupo, etc.).

In [ ]:
# 3) Filtrado: isla “Biscoe” y especie “Gentoo”
# =========================================
# 3) FILTROS (ISLA Y ESPECIE)
# =========================================
df_biscoe = df_clean.filter(F.col("island") == "Biscoe")
print("Registros en isla Biscoe:", df_biscoe.count())
#F.col("island") crea una referencia de columna (forma segura/expresiva de referirse a columnas).
# .filter(condición) devuelve un nuevo DataFrame solo con las filas que cumplen la condición.
# Aquí nos quedamos con las filas donde island == "Biscoe".
# count() es una acción: dispara la ejecución (Spark evalúa perezosamente) y recorre los datos para contar. Útil para mostrar a los estudiantes que
#  “filtro + conteo” funciona.

df_gentoo_biscoe = df_biscoe.filter(F.col("species") == "Gentoo")
print("Gentoo en Biscoe:", df_gentoo_biscoe.count())
df_gentoo_biscoe.show(5, truncate=False)
# Encadenamos otro filtro sobre el resultado anterior: ahora species == "Gentoo".
# De nuevo count() (acción) para ver cuántos quedan.
# show(5, truncate=False) muestra las primeras 5 filas completas (sin cortar texto).

Registros en isla Biscoe: 163
Gentoo en Biscoe: 119
+-------+------+--------------+-------------+-----------------+-----------+------+
|species|island|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|sex   |
+-------+------+--------------+-------------+-----------------+-----------+------+
|Gentoo |Biscoe|46.1          |13.2         |211              |4500       |FEMALE|
|Gentoo |Biscoe|50.0          |16.3         |230              |5700       |MALE  |
|Gentoo |Biscoe|48.7          |14.1         |210              |4450       |FEMALE|
|Gentoo |Biscoe|50.0          |15.2         |218              |5700       |MALE  |
|Gentoo |Biscoe|47.6          |14.5         |215              |5400       |MALE  |
+-------+------+--------------+-------------+-----------------+-----------+------+
only showing top 5 rows



Registros en isla Biscoe: 163
— De las 333 filas limpias, 163 corresponden a pingüinos muestreados en la isla Biscoe.

Gentoo en Biscoe: 119
— Dentro de Biscoe, 119 son de la especie Gentoo.
— Proporción ≈ 119 / 163 ≈ 73% → la mayoría de los registros de Biscoe son Gentoo.

Muestra de 5 filas (todas Gentoo–Biscoe)

species island bill_length bill_depth flipper_length body_mass sex
Gentoo  Biscoe     46.1       13.2         211         4500    FEMALE
Gentoo  Biscoe     50.0       16.3         230         5700    MALE
Gentoo  Biscoe     48.7       14.1         210         4450    FEMALE
Gentoo  Biscoe     50.0       15.2         218         5700    MALE
Gentoo  Biscoe     47.6       14.5         215         5400    MALE


— Es solo una vista de ejemplo (las primeras 5 filas del subconjunto).
— Observa que longitud de aleta (210–230 mm) y masa (4450–5700 g) están por encima de las medias globales que viste antes (≈ 201 mm y ≈ 4207 g). Esto cuadra con que Gentoo suele ser más grande que otras especies del dataset.

In [ ]:
# 4) Agrupaciones: medias por especie e isla; conteos por especie/sexo
# =========================================
# 4) AGRUPACIONES Y AGREGACIONES
# =========================================
# Medidas promedio por especie e isla
promedios = (df_clean
    .groupBy("species","island")
    .agg(
        F.round(F.avg("bill_length_mm"),2).alias("prom_bill_len"),
        F.round(F.avg("bill_depth_mm"),2).alias("prom_bill_dep"),
        F.round(F.avg("flipper_length_mm"),2).alias("prom_flipper"),
        F.round(F.avg("body_mass_g"),2).alias("prom_masa")
    )
    .orderBy("species","island")
)
promedios.show(20, truncate=False)
# groupBy("species","island"): crea grupos para cada combinación de especie–isla.
# .agg(...): calcula estadísticos por grupo. Aquí:
# F.avg(...) = media aritmética de cada medida biométrica.
# F.round(..., 2) = redondea a 2 decimales (estético para reporte).
# .alias(...) = nombre “bonito” para las columnas resultantes.
# .orderBy("species","island"): ordena la tabla resultado (implica shuffle; en datasets enormes puede costar).
# .show(20, truncate=False): acción que dispara la ejecución y muestra hasta 20 filas sin truncar texto.

# Conteo por especie y sexo
conteos = (df_clean
    .groupBy("species","sex")
    .count()
    .withColumnRenamed("count","n")
    .orderBy("species","sex")
)
conteos.show()
# groupBy("species","sex"): agrupa por especie y sexo.
# .count(): cuenta cuántas filas hay en cada grupo.
# .withColumnRenamed("count","n"): renombra a n para que quede más claro.
# .orderBy(...) y .show() como antes.

+---------+---------+-------------+-------------+------------+---------+
|species  |island   |prom_bill_len|prom_bill_dep|prom_flipper|prom_masa|
+---------+---------+-------------+-------------+------------+---------+
|Adelie   |Biscoe   |38.98        |18.37        |188.8       |3709.66  |
|Adelie   |Dream    |38.52        |18.24        |189.93      |3701.36  |
|Adelie   |Torgersen|39.04        |18.45        |191.53      |3708.51  |
|Chinstrap|Dream    |48.83        |18.42        |195.82      |3733.09  |
|Gentoo   |Biscoe   |47.57        |15.0         |217.24      |5092.44  |
+---------+---------+-------------+-------------+------------+---------+

+---------+------+---+
|  species|   sex|  n|
+---------+------+---+
|   Adelie|FEMALE| 73|
|   Adelie|  MALE| 73|
|Chinstrap|FEMALE| 34|
|Chinstrap|  MALE| 34|
|   Gentoo|FEMALE| 58|
|   Gentoo|  MALE| 61|
+---------+------+---+



1) Promedios por especie × isla
+---------+---------+-------------+-------------+------------+---------+
|species  |island   |prom_bill_len|prom_bill_dep|prom_flipper|prom_masa|
+---------+---------+-------------+-------------+------------+---------+
|Adelie   |Biscoe   |38.98        |18.37        |188.8       |3709.66  |
|Adelie   |Dream    |38.52        |18.24        |189.93      |3701.36  |
|Adelie   |Torgersen|39.04        |18.45        |191.53      |3708.51  |
|Chinstrap|Dream    |48.83        |18.42        |195.82      |3733.09  |
|Gentoo   |Biscoe   |47.57        |15.0         |217.24      |5092.44  |
+---------+---------+-------------+-------------+------------+---------+


Qué mide cada columna:

prom_bill_len = longitud media del pico (mm).

prom_bill_dep = profundidad media del pico (mm).

prom_flipper = longitud media de aleta (mm).

prom_masa = masa corporal media (g).

Lectura biológica rápida:

Gentoo (Biscoe): los valores más altos en aleta (~217 mm) y masa (~5092 g) ⇒ especie más grande/pesada del dataset.

Chinstrap (Dream): pico más largo (~48.83 mm) y profundidad de pico similar a Adelie (~18.4 mm), pero masa moderada (~3733 g).

Adelie (en tres islas): medias muy parecidas entre Biscoe, Dream y Torgersen; pequeñas variaciones (p. ej., aleta un poco mayor en Torgersen ~191.5 mm).

Distribución por isla (implícita en el dataset):

Gentoo aparece en Biscoe.

Chinstrap en Dream.

Adelie en las tres (Biscoe, Dream, Torgersen).

2) Conteos por especie × sexo
+---------+------+---+
|  species|   sex|  n|
+---------+------+---+
|   Adelie|FEMALE| 73|
|   Adelie|  MALE| 73|
|Chinstrap|FEMALE| 34|
|Chinstrap|  MALE| 34|
|   Gentoo|FEMALE| 58|
|   Gentoo|  MALE| 61|
+---------+------+---+


Qué es: cantidad de registros (n) por especie y sexo (después de limpiar nulos).

Lectura:

Adelie: perfectamente balanceado (73 F / 73 M) → 146 en total.

Chinstrap: también balanceado (34 / 34) → 68 en total.

Gentoo: leve sesgo a masculino (58 F / 61 M) → 119 en total.

Chequeo: 146 + 68 + 119 = 333, coincide con las filas después de limpieza.

**Conclusiones:**

Gentoo es claramente la más grande/pesada (mayor aleta y masa).

Chinstrap destaca por pico largo; masa intermedia.

Adelie muestra consistencia entre islas (diferencias pequeñas).

El balance por sexo es bueno (especialmente en Adelie y Chinstrap), lo que ayuda a evitar sesgos en comparaciones.

In [ ]:
# 5) Crear categorías de “tamaño” (ligero/medio/pesado) y promedios por categoría
# =========================================
# 5) FEATURE ENGINEERING: CATEGORÍAS
# =========================================
# Bins de masa corporal (en gramos)
# Crear la variable categórica
df_cat = df_clean.withColumn(
    "tamano",
    F.when(F.col("body_mass_g") < 3800, "ligero")
     .when((F.col("body_mass_g") >= 3800) & (F.col("body_mass_g") < 4500), "medio")
     .otherwise("pesado")
)
# withColumn("tamano", ...) añade una columna nueva al DataFrame.
# F.when(condición, valor) define ramas condicionales (como IF).
# Reglas (mutuamente excluyentes y exhaustivas):
# ligero: body_mass_g < 3800
# medio: 3800 ≤ body_mass_g < 4500
# pesado: body_mass_g ≥ 4500 (capturado por otherwise)

# Promedio de longitud de aleta por categoría
# Agregar por categoría y calcular promedios
res_tamano = (df_cat
    .groupBy("tamano")
    .agg(
        F.round(F.avg("flipper_length_mm"),2).alias("prom_flipper"),
        F.round(F.avg("body_mass_g"),2).alias("prom_masa")
    )
    .orderBy("tamano")
)
res_tamano.show()
# groupBy("tamano"): agrupa las filas por cada categoría creada.
# .agg(...): calcula agregados por grupo:
# avg("flipper_length_mm") → longitud media de aleta por categoría.
# avg("body_mass_g") → masa media (coherencia del binning).
# round(..., 2) redondea para presentación.
# orderBy("tamano"): ordena alfabéticamente ligero → medio → pesado (si quieres un orden distinto, puedes mapear a un índice y ordenar por él).

+------+------------+---------+
|tamano|prom_flipper|prom_masa|
+------+------------+---------+
|ligero|       189.3|   3419.4|
| medio|      197.73|  4096.51|
|pesado|      216.27|  5152.61|
+------+------------+---------+



Resumen por categorías de tamaño (que creamos a partir de body_mass_g):
¿Qué significa cada fila?
tamano: la clase derivada de la masa corporal
ligero < 3800 g
medio 3800–4499 g
pesado ≥ 4500 g
prom_flipper: promedio de la longitud de aleta (mm) dentro de esa categoría.
prom_masa: promedio de la masa (g) dentro de esa categoría (coherente con el binning).

Hay un patrón monótono: a mayor categoría de tamaño, mayor longitud de aleta y mayor masa.
Incrementos aproximados:
Aleta: ligero→medio +8.43 mm; medio→pesado +18.54 mm
Masa: ligero→medio +677 g; medio→pesado +1056 g

Interpretación:
La longitud de aleta y la masa se mueven juntas: cuanto más “pesado” el pingüino, más larga la aleta (consistente con la correlación positiva que suele observarse entre ambas variables).

Biológicamente, esto encaja con que especies como Gentoo (que caen mucho en “pesado”) son más grandes.

In [ ]:
# 6) Correlación: flipper_length_mm ↔ body_mass_g
# 6) CORRELACIÓN
# =========================================
corr = df_clean.stat.corr("flipper_length_mm","body_mass_g")
# Qué hace: calcula el coeficiente de correlación de Pearson entre las columnas numéricas flipper_length_mm (longitud de aleta) y body_mass_g (masa).
# Pearson mide relación lineal y devuelve un número entre −1 y 1:
# +1: relación lineal positiva perfecta (cuando una sube, la otra sube proporcionalmente).
# 0: no hay relación lineal (puede haber relación no lineal).
# −1: relación lineal negativa perfecta.
print("📊 Correlación (flipper_length_mm, body_mass_g):", round(corr,3))

📊 Correlación (flipper_length_mm, body_mass_g): 0.873


In [ ]:
# 7) Filtro por “especificaciones biológicas”: masa > 4500 g y aleta > 200 mm
# =========================================
# 7) FILTRO “ESPECIFICACIONES”
# =========================================
df_power = df_clean.filter(
    (F.col("body_mass_g") > 4500) &
    (F.col("flipper_length_mm") > 200)
)
# df_clean.filter(...) devuelve un nuevo DataFrame con las filas que cumplen la condición.
# F.col("body_mass_g") > 4500 → masa mayor a 4500 g.
# F.col("flipper_length_mm") > 200 → aleta mayor a 200 mm.
# & es el AND lógico de columnas (¡no uses and en Spark!).
# Los paréntesis son obligatorios por la precedencia de operadores.
# Nota: como partimos de df_clean, ya no hay nulos; si trabajases con nulos, podrías añadir .isNotNull().

print("Pingüinos pesados y con aleta larga:", df_power.count())
df_power.select("species","island","sex","flipper_length_mm","body_mass_g").show(10)
# .select(...) muestra solo esas columnas de interés.
# .show(10) imprime las primeras 10 filas del subconjunto.

# Por qué esos umbrales
# En el resumen previo, la media era ≈ 200.97 mm (aleta) y 4207 g (masa). Poner > 200 y > 4500 selecciona ejemplares por encima del
# promedio en ambas medidas (típicamente muchos Gentoo).

Pingüinos pesados y con aleta larga: 106
+---------+------+------+-----------------+-----------+
|  species|island|   sex|flipper_length_mm|body_mass_g|
+---------+------+------+-----------------+-----------+
|   Adelie|Biscoe|  MALE|              203|       4725|
|Chinstrap| Dream|  MALE|              205|       4550|
|Chinstrap| Dream|  MALE|              210|       4800|
|   Gentoo|Biscoe|  MALE|              230|       5700|
|   Gentoo|Biscoe|  MALE|              218|       5700|
|   Gentoo|Biscoe|  MALE|              215|       5400|
|   Gentoo|Biscoe|FEMALE|              210|       4550|
|   Gentoo|Biscoe|FEMALE|              211|       4800|
|   Gentoo|Biscoe|  MALE|              219|       5200|
|   Gentoo|Biscoe|  MALE|              215|       5150|
+---------+------+------+-----------------+-----------+
only showing top 10 rows



“Pingüinos pesados y con aleta larga: 106”
Son los registros que cumplen simultáneamente:
body_mass_g > 4500 y flipper_length_mm > 200.
Partiendo de 333 filas limpias, 106 es ~31.8% del total (106/333).

Tabla mostrada (primeras 10 filas)
Son solo los primeros 10 del subconjunto (no está ordenado a menos que tú lo pidas).
Columnas:
species (especie), island (isla), sex (sexo)
flipper_length_mm (longitud de aleta, mm)
body_mass_g (masa corporal, g)

Qué se aprecia en la muestra:
Aparecen Gentoo (Biscoe) repetidamente, con aletas de 210–230 mm y masas 5150–5700 g → coherente con que Gentoo suele ser la más grande/pesada del dataset.

Hay Chinstrap (Dream) con 205–210 mm y 4550–4800 g (también superan el umbral).
Incluso hay Adelie (Biscoe) MALE con 203 mm y 4725 g: aunque Adelie promedio es menor, algunos machos superan ambos cortes.

Se ven ambos sexos; en general, los machos tienden a aparecer más en los tramos altos de masa/alet a.

Detalles sutiles importantes
Los cortes son estrictos: > (excluye exactamente 4500 g o 200 mm).

In [ ]:
# 8) (Opcional) Modelo MLlib: predecir masa a partir de aleta y pico
# 8) MLLIB (OPCIONAL): REGRESIÓN LINEAL
# Este bloque entrena un modelo de regresión lineal en Spark MLlib para predecir la masa (body_mass_g) usando longitud de aleta y medidas del pico.
# =========================================
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
# VectorAssembler: combina varias columnas numéricas en una sola columna vector llamada típicamente "features", que es el formato que esperan los
# modelos de Spark.
# LinearRegression: modelo de regresión lineal de MLlib.

# Features y label
feat_cols = ["flipper_length_mm","bill_length_mm","bill_depth_mm"]
assembler = VectorAssembler(inputCols=feat_cols, outputCol="features")
df_ml = assembler.transform(df_clean.select(*feat_cols, "body_mass_g").dropna())
# feat_cols: las variables explicativas (X) que usaremos.
# df_clean.select(*feat_cols, "body_mass_g"): nos quedamos con X y la variable objetivo (y) a predecir.
# .dropna(): aseguramos que no haya nulos en ninguna de esas columnas (requisito para entrenar).
# assembler.transform(...): crea una nueva columna features con un vector por fila, p.ej. [flipper, bill_len, bill_dep].
# Resultado: df_ml tendrá al menos dos columnas clave:
# features (vector denso con 3 números)
# body_mass_g (label/objetivo)

# Entrenar modelo
lr = LinearRegression(featuresCol="features", labelCol="body_mass_g")
modelo = lr.fit(df_ml)
# Indicas al estimador qué columna es features y cuál es la label (por defecto sería "label", aquí lo cambiamos a "body_mass_g").
# .fit(df_ml): entrena el modelo (encuentra intercepto y coeficientes que minimizan el error cuadrático).

print("Intercepto:", round(modelo.intercept,2))
print("Coeficientes:", [round(c,3) for c in modelo.coefficients])
print("R2:", round(modelo.summary.r2,3))
# Intercepto: valor de masa estimada cuando todas las X valen 0 (aquí es interpretación matemática; fisiológicamente 0 mm no tiene sentido, pero el intercepto
# ayuda al ajuste).
# Coeficientes: cuánto cambia la masa (en gramos) cuando aumenta 1 unidad una de las X manteniendo las demás constantes.

# Predicciones de muestra
modelo.transform(df_ml).select("features","body_mass_g","prediction").show(5, truncate=False)
# transform añade la columna prediction (masa estimada).
# El select muestra lado a lado: X (features), y real (body_mass_g) y ŷ (prediction) para 5 filas.

Intercepto: -6445.48
Coeficientes: [np.float64(50.762), np.float64(3.293), np.float64(17.836)]
R2: 0.764
+-----------------+-----------+------------------+
|features         |body_mass_g|prediction        |
+-----------------+-----------+------------------+
|[181.0,39.1,18.7]|3750       |3204.7612272860642|
|[186.0,39.5,17.4]|3800       |3436.7017222980267|
|[195.0,40.3,18.0]|3250       |3906.897031997756 |
|[193.0,36.7,19.3]|3450       |3816.7057718756096|
|[190.0,39.3,20.6]|3650       |3696.1681278223004|
+-----------------+-----------+------------------+
only showing top 5 rows



Es una regresión lineal que predice body_mass_g (masa en gramos) a partir de:

flipper_length_mm (aleta, mm)

bill_length_mm (largo del pico, mm)

bill_depth_mm (profundidad del pico, mm)

La ecuación estimada es:
masa=−6445.48+50.762⋅aleta+3.293⋅pico_largo+17.836⋅pico_profundo

Intercepto: -6445.48

Es el valor que toma la predicción cuando todas las X valen 0.

Aquí no tiene interpretación física (0 mm de aleta/pico no existe); actúa como término de ajuste para que el plano se coloque donde mejor minimiza el error.

Coeficientes: [50.762, 3.293, 17.836]
Ordenados como tus features: [flipper_length_mm, bill_length_mm, bill_depth_mm].

Interpretación marginal (manteniendo las otras constantes):
Aleta: +1 mm ≈ +50.762 g de masa.
Pico (largo): +1 mm ≈ +3.293 g.
Pico (profundidad): +1 mm ≈ +17.836 g.
El más influyente es la longitud de aleta (tiene el mayor peso).

R²: 0.764
El modelo explica el 76.4% de la variabilidad de la masa (con estas 3 variables lineales).
Es bueno para un modelo simple; ~23.6% queda sin explicar (especies/sexo/dieta/ruido, relaciones no lineales, etc.).

In [ ]:
# 9) Exportar resultados a un único CSV (robusto)
# =========================================
# 9) EXPORTACIÓN A UN ÚNICO CSV
# =========================================
# Elegimos qué resultado exportar (puedes cambiarlo por 'promedios' o 'conteos')
a_exportar = df_power
# Seleccionas el DataFrame que quieres exportar. Puedes reemplazar df_power por cualquier otro (p. ej., promedios, conteos).

# Escribir como carpeta temporal con un solo 'part-*.csv'
out_dir = "/content/penguins_resultados_out"
a_exportar.coalesce(1).write.mode("overwrite").option("header", True).csv(out_dir)
# coalesce(1): reduce a 1 partición, así Spark escribe un solo part-*.csv.
# Nota: coalesce(1) es costoso si el DF es grande (movimiento de datos). Úsalo solo si necesitas de verdad un único archivo.
# mode("overwrite"): borra la carpeta si ya existía.
# option("header", True): incluye encabezados en el CSV.
# .csv(out_dir): Spark siempre escribe una carpeta (out_dir) que contiene el(s) archivo(s) part-*.csv y metadatos _SUCCESS.

# Mover/renombrar al archivo final 'penguins_resultados.csv'
import os, glob, shutil
dest = "/content/penguins_resultados.csv"
# dest es el nombre final que quieres (un archivo, no carpeta).

# Limpiar si quedó una carpeta/archivo previo con el mismo nombre
if os.path.exists(dest):
    if os.path.isdir(dest):
        shutil.rmtree(dest)
    else:
        os.remove(dest)
# Evita el error IsADirectoryError que ya viste: si dest existía como carpeta de intentos anteriores, hay que borrarla de forma distinta que un archivo.

part = glob.glob(os.path.join(out_dir, "part-*.csv"))
if not part:
    raise FileNotFoundError("No se encontró part-*.csv en " + out_dir)
# Busca el único CSV que Spark generó dentro de out_dir.
# Si no existe, se lanza un error explicativo (descarga fallida, permisos, etc.).

shutil.move(part[0], dest)
shutil.rmtree(out_dir, ignore_errors=True)
# Mueve/renombra el part-*.csv al nombre final dest.
# Elimina la carpeta temporal out_dir para dejar todo limpio.

print("✅ Archivo final listo en:", dest)
# Mensaje final con la ruta del CSV único.

✅ Archivo final listo en: /content/penguins_resultados.csv


**Segundo Ejemplo**
¿Cuál es el objetivo del ejemplo y por qué “es de Big Data”?
Objetivo del ejemplo

Este mini Word Count con RDDs busca que tus estudiantes entiendan, con un caso súper simple, los conceptos base de computación distribuida en Spark:

RDD (Resilient Distributed Dataset)

Qué es un dataset inmutable y particionado que Spark distribuye por el clúster.

Que trabajas con particiones (no con una lista local).

Transformaciones vs Acciones (ejecución perezosa)

flatMap, map, reduceByKey = transformaciones que construyen un DAG de ejecución.

collect() = acción que dispara el cómputo.

Patrón MapReduce aplicado a texto

Map: (palabra → (palabra,1))

Reduce: agrupar por clave y sumar las ocurrencias (reduceByKey).

Shuffle y eficiencia

reduceByKey hace combiner local antes del shuffle, moviendo menos datos por red que groupByKey.

Idea clave para escalar.

Tubería reproducible

El mismo pipeline (limpieza → tokenización → conteo → top-N) se usa en logs, clicstreams, textos, etc.

¿Por qué es “de Big Data” si el ejemplo es pequeño?

Porque enseña el modelo y la arquitectura que permiten procesar terabytes aunque aquí uses 3 líneas para que el resultado sea legible. Lo “Big Data” no es el tamaño del ejemplo, sino:

Abstracciones distribuidas (RDD/particiones) y ejecución paralela.

Planificación de shuffles, combiners, tolerancia a fallos.

Código que escala sin cambiar: lo mismo que corre en tu notebook corre en un clúster sobre HDFS/S3.

Si mañana cambias sc.parallelize(texto) por sc.textFile("s3://bucket/logs/2025/*") y quitas collect(), el pipeline procesa miles de archivos en paralelo con el mismo patrón.

Qué se aprende?

Se aprende el esqueleto del cálculo distribuido: particionar → mapear → barajar/agrupar → reducir.

Entienden dónde está el costo (el shuffle) y cómo elegir primitivas más eficientes.

Ven la diferencia entre “prototipo chico” y “ejecución a escala”: mismo código, distintas fuentes/volúmenes.

Cómo “vestirlo” de Big Data en clase (rápido)

Reemplazar la fuente:

rdd = sc.textFile("s3://mi-bucket/logs/*.gz")  # o HDFS


Evitar traer todo al driver:

conteo.saveAsTextFile("s3://mi-bucket/out/wordcount/")


Mostrar el DAG en la Spark UI (Stages/Tasks) y explicar el shuffle entre map y reduceByKey.

Con esto, el ejercicio deja de ser solo “contar palabras” y se convierte en una demostración guiada de los fundamentos de Big Data con Spark.

In [ ]:
# Segundo ejemplo
# 0) Instalación y sesión de Spark (una sola vez)
# --- INSTALAR Y CREAR LA SESIÓN DE SPARK ---
!pip -q install pyspark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Clase-BigData-PySpark") \
    .master("local[*]") \
    .getOrCreate()
# SparkSession.builder inicia el constructor de la sesión.
# .appName("Clase-BigData-PySpark") pone un nombre a tu aplicación (aparece en logs/UI).
# .master("local[*]") indica dónde correr:
# local[*] = modo local usando todos los núcleos disponibles.

print("Versión de Spark:", spark.version)

Versión de Spark: 3.5.1


In [ ]:
# 1) RDDs: map, filter, reduce (Word Count mini)
# --- RDDs: TRANSFORMACIONES Y ACCIONES ---
sc = spark.sparkContext
# sc es el SparkContext: la puerta de entrada al API de RDD (bajo nivel).

texto = [
    "big data es asombroso",
    "spark hace big data mas rapido",
    "data ciencia y big data"
]
# Pequeño corpus en memoria (3 líneas).

# Crear RDD
rdd = sc.parallelize(texto)
# parallelize reparte esa lista entre particiones → obtenemos un RDD de 3 elementos (una línea por elemento).
# RDD = Resilient Distributed Dataset (inmutable, distribuido).

# Limpieza simple, dividir en palabras y contar
palabras = rdd.flatMap(lambda linea: linea.lower().split())
# Transformación (flatMap): por cada línea, la pasa a minúsculas y la divide en palabras con split().
# flatMap “aplana”: una línea produce varias salidas (las palabras), y todas se unen en un solo RDD.
# Resultado: RDD de strings: ["big","data","es","asombroso", ...].

# (palabra, 1)
pares = palabras.map(lambda w: (w, 1))
# Transformación (map): convierte cada palabra en un par clave-valor (palabra, 1).
# Es el clásico patrón MapReduce para conteo.

# reducir por clave
conteo = pares.reduceByKey(lambda a, b: a + b)
# Transformación de agregación por clave: suma los 1 de cada palabra.
# Importante: reduceByKey hace combiner local antes del shuffle → menos datos por red que con groupByKey.
# Resultado: RDD de (palabra, total).

print("=== CONTEO DE PALABRAS ===")
for palabra, n in conteo.collect():
    print(f"{palabra}: {n}")
# Acción (collect): trae todo al driver como lista de Python y lo imprime.
# Úsalo solo si el resultado cabe en memoria; en datos grandes, mejor .take(k) o escribir a disco.
# Con tu corpus, los totales esperados son: data: 4, big: 3, el resto 1.

# TOP 3 más frecuentes
top3 = conteo.takeOrdered(3, key=lambda x: -x[1])
print("\nTop 3:", top3)
# Acción (takeOrdered): devuelve los 3 primeros según el orden definido por la función key.
# Aquí ordenas por -x[1] (el negativo del conteo) ⇒ equivalente a ordenar descendente por frecuencia.
# Resultado típico: [('data', 4), ('big', 3), ('... ', 1)] (el tercero puede variar si hay empates con 1).

=== CONTEO DE PALABRAS ===
big: 3
es: 1
asombroso: 1
hace: 1
mas: 1
data: 4
spark: 1
rapido: 1
ciencia: 1
y: 1

Top 3: [('data', 4), ('big', 3), ('es', 1)]


Partiste de 3 líneas:

big data es asombroso

spark hace big data mas rapido

data ciencia y big data

Tras lower() y split(), el conteo queda:

data: 4 apariciones (1ª línea: 1; 2ª: 1; 3ª: 2 → al inicio y al final).

big: 3 (una en cada línea).

Todas las demás aparecen 1 vez: es, asombroso, hace, mas, spark, rapido, ciencia, y.

El orden en el bloque de conteos no está ordenado (viene de collect()), por eso data: 4 te sale en medio y no arriba.

Top 3
Top 3: [('data', 4), ('big', 3), ('es', 1)]


Usaste takeOrdered(3, key=lambda x: -x[1]), que ordena por frecuencia descendente.

Los dos primeros son claros: ('data', 4) y ('big', 3).

El tercer lugar está empatado (todas las demás valen 1). Apareció ('es', 1), pero podría ser cualquiera de las que tienen 1, porque:

Hay empate y el orden de desempate no está garantizado (no es determinista).

In [ ]:
# Otros ejemplos
# 2) DataFrames básicos: schema, select, filter, groupBy, SQL
# para mostrar la API de DataFrames/SQL. Puedes ejecutarlo en el mismo notebook, pero es independiente: solo necesita que ya exista la sesión spark.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
# functions as F te da expresiones de columna (sum, avg, col, round, etc.).
# types te permite definir un esquema (nombres y tipos de columnas)

# --- CREAR UN DATAFRAME DE VENTAS EN MEMORIA ---
# Qué hace, en breve
# Define un esquema y crea un DataFrame en memoria (no lee archivos)
schema = StructType([
    StructField("producto", StringType(), True),
    StructField("fecha",    StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio",   DoubleType(), True),
])
# Estás declarando el esquema: cada StructField = (nombre, tipo, nullable).
# Nota: fecha aquí es String; si luego quieres operar con fechas, conviene convertirla a DateType.
# (producto, fecha, cantidad, precio_unitario)

data = [
    ("Producto A", "2025-01-01", 10, 15.5),
    ("Producto B", "2025-01-02",  5, 25.0),
    ("Producto A", "2025-01-03", 12, 15.5),
    ("Producto C", "2025-01-01",  8, 40.0),
    ("Producto B", "2025-01-03",  6, 25.0),
]
# Lista de tuplas: cada tupla es una fila.

df = spark.createDataFrame(data, schema)
df.printSchema()
df.show()
# Construye el DataFrame distribuido.
# printSchema() confirma que Spark entendió los tipos.
# show() enseña las filas (acción que ejecuta).

# --- SELECCIÓN Y FILTRO ---
df_sel = df.select("producto", "fecha", "cantidad", "precio") \
           .where(F.col("cantidad") >= 8)
print("Filtrados con cantidad >= 8:")
df_sel.show()
# select(...) se queda con esas columnas.
# .where(...) (igual que .filter(...)) aplica el predicado cantidad >= 8.
# Con tus datos, pasan:
# ("A",10,15.5), ("A",12,15.5), ("C",8,40.0) → 3 filas.

# --- AGREGACIÓN (ventas totales por producto) ---
df_ingresos = df.withColumn("ingresos", F.col("cantidad") * F.col("precio")) \
                .groupBy("producto") \
                .agg(
                    F.sum("cantidad").alias("cantidad_total"),
                    F.round(F.sum("ingresos"), 2).alias("ingresos_totales")
                ) \
                .orderBy(F.desc("ingresos_totales"))
# Paso a paso:
# withColumn("ingresos", cantidad*precio) crea la columna monto por fila.
# groupBy("producto") agrupa por producto.
# agg(...) calcula dos agregados por grupo:
# sum(cantidad) → unidades totales.
# sum(ingresos) → dinero total, redondeado a 2 decimales.
# orderBy(desc(...)) ordena de mayor a menor por ingresos.

# Producto A: (10×15.5) + (12×15.5) = 155 + 186 = 341; cantidad = 22
# Producto B: (5×25) + (6×25) = 125 + 150 = 275; cantidad = 11
# Producto C: (8×40) = 320; cantidad = 8
# Ordenado por ingresos: A (341), C (320), B (275).

print("Ingresos por producto:")
df_ingresos.show()

# --- USAR SQL TEMP VIEW ---
df.createOrReplaceTempView("ventas")
consulta = spark.sql("""
    SELECT producto,
           SUM(cantidad) AS cantidad_total,
           ROUND(SUM(cantidad*precio), 2) AS ingresos_totales
    FROM ventas
    GROUP BY producto
    ORDER BY ingresos_totales DESC
""")
print("Consulta SQL:")
consulta.show()
# Registras df como vista temporal llamada ventas.
# Haces la misma agregación pero en SQL (útil si tus alumnos vienen de SQL).
# consulta y df_ingresos deben dar lo mismo.


# --- EXPLICAR PLAN (lazy evaluation / DAG) ---
print("Plan lógico/físico de la agregación:")
df_ingresos.explain()
# Imprime el plan lógico y el plan físico que Spark ejecutará (proyecciones, agregaciones, shuffles, etc.).
# No ejecuta el job (muestra cómo se ejecutaría cuando llegue una acción como show, count, write).
# Es perfecto para ilustrar la evaluación perezosa y el optimizador Catalyst.

root
 |-- producto: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)

+----------+----------+--------+------+
|  producto|     fecha|cantidad|precio|
+----------+----------+--------+------+
|Producto A|2025-01-01|      10|  15.5|
|Producto B|2025-01-02|       5|  25.0|
|Producto A|2025-01-03|      12|  15.5|
|Producto C|2025-01-01|       8|  40.0|
|Producto B|2025-01-03|       6|  25.0|
+----------+----------+--------+------+

Filtrados con cantidad >= 8:
+----------+----------+--------+------+
|  producto|     fecha|cantidad|precio|
+----------+----------+--------+------+
|Producto A|2025-01-01|      10|  15.5|
|Producto A|2025-01-03|      12|  15.5|
|Producto C|2025-01-01|       8|  40.0|
+----------+----------+--------+------+

Ingresos por producto:
+----------+--------------+----------------+
|  producto|cantidad_total|ingresos_totales|
+----------+--------------+----------------+
|Producto

root
 |-- producto: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)
Son los tipos de columna que definiste en el schema.

nullable = true indica que la columna acepta NULL.

fecha quedó como String (si quieres operar por fechas, conviene castear a date).

Tabla mostrada:

python-repl
Copiar código
+----------+----------+--------+------+
|producto  |fecha     |cantidad|precio|
...
Es simplemente el contenido del DataFrame que creaste en memoria (no viene de archivo).

2) Filtro cantidad >= 8
diff
Copiar código
Filtrados con cantidad >= 8:
+----------+----------+--------+------+
|Producto A|2025-01-01|      10|  15.5|
|Producto A|2025-01-03|      12|  15.5|
|Producto C|2025-01-01|       8|  40.0|
Muestra solo las filas donde cantidad es 8 o más (A: 10 y 12; C: 8).

3) Agregación “ingresos por producto”
diff
Copiar código
+----------+--------------+----------------+
|producto  |cantidad_total|ingresos_totales|
+----------+--------------+----------------+
|Producto A|            22|           341.0|
|Producto C|             8|           320.0|
|Producto B|            11|           275.0|
Cálculos:

A: (10×15.5) + (12×15.5) = 341.0; cantidad_total = 22

C: (8×40.0) = 320.0; cantidad_total = 8

B: (5×25.0) + (6×25.0) = 275.0; cantidad_total = 11

Está ordenado por ingresos_totales descendente.

La sección “Consulta SQL” devuelve lo mismo pero usando la vista temporal ventas y SQL.

4) explain() — Plan físico (cómo Spark ejecuta)
pgsql
Copiar código
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [ingresos_totales DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(ingresos_totales DESC NULLS LAST, 200), ENSURE_REQUIREMENTS
      +- HashAggregate(keys=[producto], functions=[sum(cantidad), sum(ingresos)])
         +- Exchange hashpartitioning(producto, 200), ENSURE_REQUIREMENTS
            +- HashAggregate(keys=[producto], functions=[partial_sum(cantidad), partial_sum(ingresos)])
               +- Project [producto, cantidad, (cast(cantidad as double) * precio) AS ingresos]
                  +- Scan ExistingRDD[producto,fecha,cantidad,precio]
Léelo de abajo hacia arriba:

Scan ExistingRDD

Está “leyendo” el DataFrame que creaste desde memoria (por eso “ExistingRDD”; no hay lectura de archivo).

Project

Calcula la columna ingresos = cantidad * precio.

Nota el cast(cantidad as double): como precio es double y cantidad int, Spark promueve a double para multiplicar en coma flotante.

HashAggregate (partial_sum)

Hace agregación parcial por partición: suma cantidad e ingresos dentro de cada partición (combiner local).

Esto reduce datos antes de barajar: es más eficiente que mandar todo crudo.

Exchange hashpartitioning(producto, 200)

Shuffle: rebaraja los datos por la clave producto a 200 particiones (valor por defecto de spark.sql.shuffle.partitions).

Así, todas las filas del mismo producto terminan juntas para poder cerrar la agregación.

HashAggregate (final)

Cierra la agregación: suma los parciales de cada producto y obtiene los totales finales.

Exchange rangepartitioning(ingresos_totales DESC, 200)

Otro shuffle para poder ordenar globalmente por ingresos_totales de mayor a menor (rangepartitioning facilita el sort total).

Sort

Orden global final (NULLS LAST significa que, si hubiera nulos, irían al final).

AdaptiveSparkPlan isFinalPlan=false

Adaptive Query Execution está activo: Spark puede ajustar el plan en tiempo de ejecución (p. ej., fusionar particiones, manejar skew). “isFinalPlan=false” indica que el plan podría adaptarse durante la ejecución real.

In [ ]:
# Otro ejemplo
# 3) Leer/guardar CSV con Spark (creando el CSV en Colab)
# este bloque enseña cómo crear un CSV “de juguete” con pandas, leerlo con Spark, agregar/agrupar y guardar el resultado.
import pandas as pd

# --- CREAR Y GUARDAR UN CSV LOCAL ---
ventas_pd = pd.DataFrame({
    "producto": ["A","B","A","C","B","D","A"],
    "fecha":    ["2025-03-01","2025-03-01","2025-03-02","2025-03-02","2025-03-03","2025-03-03","2025-03-03"],
    "cantidad": [ 10, 5, 12,  7,  3, 11,  2 ],
    "precio":   [15.5,25.0,15.5,40.0,25.0,19.0,15.5]
})
ventas_pd.to_csv("/content/ventas.csv", index=False)
print("Archivo escrito en /content/ventas.csv")
# Se arma un DataFrame de pandas con ventas ficticias.
# to_csv(...) lo guarda en /content/ventas.csv (ruta típica de Colab).
# Ventaja: no dependes de subir archivos externos; generas uno al vuelo para practicar.
# Nota: index=False evita la columna de índice en el CSV.

# --- LEER CSV CON SPARK E INFERIR TIPOS ---
df_csv = spark.read.csv("/content/ventas.csv", header=True, inferSchema=True)
df_csv.printSchema()
df_csv.show()
# spark.read.csv(...) crea un DataFrame de Spark (distribuido).
# header=True indica que la primera fila tiene nombres de columnas.
# inferSchema=True deja que Spark deduzca tipos (aquí cantidad→int, precio→double).
# En producción es preferible definir el esquema para evitar sorpresas.
# printSchema() muestra tipos; show() imprime filas (y dispara la lectura porque es una acción).
# Tip: fecha queda como String; si luego filtras por mes/año, conviene convertirla:
#df_csv = df_csv.withColumn("fecha", F.to_date("fecha")).


# --- AGREGAR COLUMNA DE INGRESOS Y GUARDAR RESULTADOS ---
df_csv_res = df_csv.withColumn("ingresos", F.col("cantidad") * F.col("precio")) \
                   .groupBy("producto") \
                   .agg(
                       F.sum("cantidad").alias("cantidad_total"),
                       F.round(F.sum("ingresos"), 2).alias("ingresos_totales")
                   ) \
                   .orderBy(F.desc("ingresos_totales"))

df_csv_res.show()
# withColumn("ingresos", cantidad*precio) crea el monto por fila.
# groupBy("producto").agg(...) resume por producto:
# sum(cantidad) = unidades totales.
# sum(ingresos) = ingresos totales (redondeados a 2 decimales para presentación).
# orderBy(desc(...)) ordena de mayor a menor por ingresos.
# show() ejecuta el plan y muestra la tabla final.
# Con esos datos, el resultado esperado:
# A: cantidad 24, ingresos 372.0 (15.5×(10+12+2))
# C: cantidad 7, ingresos 280.0
# D: cantidad 11, ingresos 209.0
# B: cantidad 8, ingresos 200.0
# → Orden: A, C, D, B.


# Guardar como CSV (carpeta de salida con múltiples partes)
salida = "/content/ingresos_totales_out"
df_csv_res.coalesce(1).write.mode("overwrite").option("header", True).csv(salida)
print(f"Resultados guardados en {salida}")
# Spark siempre escribe carpetas: csv(salida) crea el directorio /content/ingresos_totales_out/ con:
# un archivo part-*.csv (tu data) y
# archivos de metadatos (_SUCCESS, etc.).
# coalesce(1) fuerza un solo part-*.csv (útil para entregar un único archivo).
# En datos grandes puede ser costoso; úsalo solo si necesitas 1 archivo.
# mode("overwrite") borra la carpeta si ya existía.
# header=True vuelve a escribir los encabezados.

Archivo escrito en /content/ventas.csv
root
 |-- producto: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)

+--------+----------+--------+------+
|producto|     fecha|cantidad|precio|
+--------+----------+--------+------+
|       A|2025-03-01|      10|  15.5|
|       B|2025-03-01|       5|  25.0|
|       A|2025-03-02|      12|  15.5|
|       C|2025-03-02|       7|  40.0|
|       B|2025-03-03|       3|  25.0|
|       D|2025-03-03|      11|  19.0|
|       A|2025-03-03|       2|  15.5|
+--------+----------+--------+------+

+--------+--------------+----------------+
|producto|cantidad_total|ingresos_totales|
+--------+--------------+----------------+
|       A|            24|           372.0|
|       C|             7|           280.0|
|       D|            11|           209.0|
|       B|             8|           200.0|
+--------+--------------+----------------+

Resultados guardados en /content/ingr

1) Archivo escrito en /content/ventas.csv

Pandas creó y guardó el CSV “de práctica” en esa ruta de Colab. Todo OK con la escritura.

2) Esquema (printSchema())
producto: string
fecha:    date
cantidad: integer
precio:   double


Tipos correctos para el análisis: texto, fecha (si aplicaste to_date o Spark la infirió), enteros y decimales.

nullable = true indica que podrían existir nulos (aunque en tu muestra no los hay).

3) Vista de datos (show())

Muestra las 7 filas del CSV:

Tres ventas de A (días 1, 2 y 3),

Dos de B (días 1 y 3),

Una de C (día 2) y una de D (día 3).
Las fechas aparecen en formato YYYY-MM-DD (tipo date).

4) Agregación “ingresos por producto”
+--------+--------------+----------------+
|producto|cantidad_total|ingresos_totales|
+--------+--------------+----------------+
|A       |24            |372.0           |
|C       |7             |280.0           |
|D       |11            |209.0           |
|B       |8             |200.0           |


Cálculos (fila a fila → sumar por producto):

A: cantidades 10 + 12 + 2 = 24; ingresos 15.5×(10+12+2) = 372.0

B: 5 + 3 = 8; ingresos 25×8 = 200.0

C: 7; ingresos 40×7 = 280.0

D: 11; ingresos 19×11 = 209.0
Ordenados descendente por ingresos_totales: A > C > D > B.

5) Resultados guardados en /content/ingresos_totales_out

Spark escribió el resultado como carpeta con:

un archivo part-*.csv (tus datos),

metadatos (como _SUCCESS).

In [ ]:
# 4) Cache & performance (demostración simple)}
# --- CACHE PARA REUSO DE RESULTADOS EN MEMORIA ---
df_grande = df_csv.withColumn("ingresos", F.col("cantidad") * F.col("precio")) \
                  .withColumn("ingresos_con_iva", F.col("ingresos") * F.lit(1.16))
# Crea un DF derivado añadiendo:
# ingresos = cantidad * precio
# ingresos_con_iva = ingresos * 1.16 (usa F.lit(1.16) para la constante IVA).
# Aún no se ejecuta nada (Spark es lazy).


# Primera acción (sin cache) — se ejecuta el plan
_ = df_grande.count()
# count() es una acción → materializa todas las transformaciones (lee, multiplica, etc.) sin caché.
# Aquí Spark recorre todos los datos por primera vez.

# Cachear y volver a ejecutar (aprovecha memoria)
df_grande.cache()
_ = df_grande.count()
# cache() marca el DF para guardarlo en memoria (nivel por defecto: MEMORY_ONLY).
# La caché se llena con la siguiente acción; por eso el segundo count():
# Esta segunda pasada materializa y guarda df_grande en RAM.
# A partir de aquí, acciones posteriores sobre df_grande leerán desde memoria (mucho más rápido) y no recalcularán todo el plan.

print("Cache aplicado. Vuelve a usar los resultados sin recalcular todo el plan.")

Cache aplicado. Vuelve a usar los resultados sin recalcular todo el plan.


In [ ]:
# 5) MLlib: Regresión lineal (pipeline simple)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
# VectorAssembler: combina varias columnas numéricas en una sola columna vector (features) que es el formato que esperan los modelos de Spark.
# LinearRegression: estimador de regresión lineal

# Dataset sintético: predecir precio por cantidad y "descuento"
data_ml = spark.createDataFrame([
    (10, 0.00, 160.0),
    ( 5, 0.10, 110.0),
    (12, 0.05, 185.0),
    ( 7, 0.20, 105.0),
    (15, 0.00, 240.0),
    ( 3, 0.15,  70.0),
    (11, 0.05, 175.0),
    ( 8, 0.10, 130.0),
    (20, 0.00, 320.0),
    ( 6, 0.10, 115.0),
], ["cantidad", "descuento", "precio_total"])
# Creas un DataFrame pequeño (sintético) con:
# cantidad (unidades),
# descuento (proporción, p. ej. 0.10 = 10%),
# precio_total (variable objetivo a predecir).

# Ensamblar features
ens = VectorAssembler(inputCols=["cantidad", "descuento"], outputCol="features")
df_feats = ens.transform(data_ml)
# Toma las columnas cantidad y descuento y crea una columna features con un vector:
# por ej., (cantidad=10, descuento=0.00) → features=[10.0, 0.0].

# Modelo lineal
lr = LinearRegression(featuresCol="features", labelCol="precio_total")
modelo = lr.fit(df_feats)
# Le dices al estimador qué columna es features (X) y cuál es la label (precio_total, y).

.fit(...) entrena la regresión lineal.

La forma del modelo que aprende es:

precio_total^=𝛽0+𝛽1⋅cantidad+𝛽2⋅descuento


Esperable:
𝛽1>0 (más cantidad ⇒ más precio total)
𝛽2<0 (más descuento ⇒ menor precio total), si todo está consistente.

print("=== Coeficientes del modelo ===")
print("Intercepto:", modelo.intercept)
print("Coeficientes [cantidad, descuento]:", modelo.coefficients.tolist())
print("R2:", round(modelo.summary.r2, 4))

# Predicciones
pred = modelo.transform(df_feats).select("cantidad","descuento","precio_total","prediction")
pred.show()

=== Coeficientes del modelo ===
Intercepto: 35.67560696580733
Coeficientes [cantidad, descuento]: [13.576967077666144, -84.96250158891918]
R2: 0.9832
+--------+---------+------------+------------------+
|cantidad|descuento|precio_total|        prediction|
+--------+---------+------------+------------------+
|      10|      0.0|       160.0|171.44527774246876|
|       5|      0.1|       110.0| 95.06419219524614|
|      12|     0.05|       185.0| 194.3510868183551|
|       7|      0.2|       105.0| 113.7218761916865|
|      15|      0.0|       240.0|239.33011313079948|
|       3|     0.15|        70.0|63.662132960467886|
|      11|     0.05|       175.0|180.77411974068895|
|       8|      0.1|       130.0|135.79509342824457|
|      20|      0.0|       320.0| 307.2149485191302|
|       6|      0.1|       115.0|108.64115927291228|
+--------+---------+------------+------------------+



1) La ecuación que aprendió

Con esos parámetros, el modelo de regresión lineal queda:

precio_total^=35.676+13.577⋅cantidad−84.963⋅descuento
precio_total

Intercepto = 35.676: es el valor base cuando cantidad=0 y descuento=0. No tiene sentido económico directo (no vendes 0 unidades), pero desplaza la recta/plano para minimizar el error global.

Coeficiente de cantidad ≈ +13.577: cada unidad adicional aumenta el precio total en ~13.58 (manteniendo el descuento fijo).

Coeficiente de descuento ≈ −84.963: como descuento está en proporción (0.10 = 10%), subir el descuento 1.0 (100 pp) restaría ~84.96.

Regla útil: 10% (0.10) ≈ −8.50, 5% (0.05) ≈ −4.25, 20% (0.20) ≈ −17.0.

El signo negativo de descuento es lo esperado: más descuento ⇒ menor precio total.

2) Calidad del ajuste

R² = 0.9832 → el 98.32% de la variación del precio_total queda explicada por el modelo.
Para un ejemplo didáctico es muy alto: el plano lineal (cantidad, descuento) captura casi todo el patrón.

3) Predicciones vs valores reales (ejemplos)

Miremos algunos pares real vs predicción y su residual (real − pred):

(10, 0.00): real 160.0 vs pred 171.45 → residual ≈ −11.45 (sobreestima).

(5, 0.10): real 110.0 vs pred 95.06 → residual ≈ +14.94 (subestima).

(12, 0.05): real 185.0 vs pred 194.35 → residual ≈ −9.35.

(15, 0.00): real 240.0 vs pred 239.33 → residual ≈ +0.67 (muy cerca).

(20, 0.00): real 320.0 vs pred 307.21 → residual ≈ +12.79 (subestima).

Es normal ver aciertos y desvíos: el modelo minimiza el error global, no acierta cada punto.

4) Interpretación rápida para clase

Cantidad domina el precio total (coeficiente grande y positivo).

Descuento reduce el total con un efecto lineal y constante (independiente de la cantidad, tal como está modelado).

El alto R² indica que, para estos datos sintéticos, una relación lineal simple funciona muy bien.

In [ ]:
# 6) Bonus: Join entre tablas (dimensión + hechos)
# Tabla de productos (dimensión)
# Este bloque muestra un join típico de modelo estrella: una tabla de hechos (ventas) enriquecida con una tabla dimensión (productos), y luego un
# reporte por categoría.
dim_prod = spark.createDataFrame([
    ("A", "Bebidas",  "ProveedorX"),
    ("B", "Snacks",   "ProveedorY"),
    ("C", "Lácteos",  "ProveedorZ"),
    ("D", "Bebidas",  "ProveedorX"),
], ["producto", "categoria", "proveedor"])
# Crea una tabla pequeña (lookup) con una fila por producto (clave), y atributos categoria y proveedor.
# Debe tener una fila por clave para no duplicar ventas al unir.


# Hechos de ventas (ya tenemos df_csv)
ventas = df_csv.withColumn("ingresos", F.col("cantidad") * F.col("precio"))
# Parte de df_csv (tus ventas leídas del CSV) y agrega ingresos = cantidad * precio.
# Esta es la tabla de hechos (muchas filas por producto/fecha).

# JOIN por clave de producto
joined = ventas.join(dim_prod, on="producto", how="left")
# on="producto": une por la columna común producto.
# how="left" (LEFT OUTER JOIN):
# Mantiene todas las filas de ventas (izquierda), aunque no haya coincidencia en dim_prod.
# Si un producto no está en la dimensión, categoria/proveedor quedarán NULL (en tu caso, A–D sí existen, así que se completan).
# Resultado: cada fila de ventas ahora trae categoria y proveedor “pegados” desde la dimensión.
# Tip de performance (como dim_prod es pequeña):
# ventas.join(F.broadcast(dim_prod), on="producto", how="left")
# reduce el shuffle y acelera el join.


# Agregar por categoría
res_cat = joined.groupBy("categoria").agg(
    F.sum("cantidad").alias("cantidad_total"),
    F.round(F.sum("ingresos"),2).alias("ingresos_totales")
).orderBy(F.desc("ingresos_totales"))
# Agrupa por categoria (venimos del join, así que cada venta ya sabe su categoría).
# Calcula:
# cantidad_total = suma de unidades.
# ingresos_totales = suma de ingresos (redondeado para el reporte).
# Ordena de mayor a menor por facturación.

joined.show()
res_cat.show()
# joined.show() → te enseña ventas enriquecidas (producto, fecha, cantidad, precio, ingresos, categoria, proveedor).
# res_cat.show() → el resumen por categoría.

+--------+----------+--------+------+--------+---------+----------+
|producto|     fecha|cantidad|precio|ingresos|categoria| proveedor|
+--------+----------+--------+------+--------+---------+----------+
|       B|2025-03-01|       5|  25.0|   125.0|   Snacks|ProveedorY|
|       B|2025-03-03|       3|  25.0|    75.0|   Snacks|ProveedorY|
|       D|2025-03-03|      11|  19.0|   209.0|  Bebidas|ProveedorX|
|       C|2025-03-02|       7|  40.0|   280.0|  Lácteos|ProveedorZ|
|       A|2025-03-01|      10|  15.5|   155.0|  Bebidas|ProveedorX|
|       A|2025-03-02|      12|  15.5|   186.0|  Bebidas|ProveedorX|
|       A|2025-03-03|       2|  15.5|    31.0|  Bebidas|ProveedorX|
+--------+----------+--------+------+--------+---------+----------+

+---------+--------------+----------------+
|categoria|cantidad_total|ingresos_totales|
+---------+--------------+----------------+
|  Bebidas|            35|           581.0|
|  Lácteos|             7|           280.0|
|   Snacks|             8|     

ingresos = cantidad * precio (columna calculada).

El LEFT JOIN por producto pegó categoria y proveedor desde la dimensión:

A y D → Bebidas / ProveedorX

B → Snacks / ProveedorY

C → Lácteos / ProveedorZ

Esta tabla sirve para ver cada transacción ya clasificada por categoría.

2) Agregado por categoría
+---------+--------------+----------------+
|categoria|cantidad_total|ingresos_totales|
+---------+--------------+----------------+
|Bebidas  |35            |581.0           |
|Lácteos  |7             |280.0           |
|Snacks   |8             |200.0           |


Cómo se obtiene:

Bebidas = productos A + D

Cantidad: A (10+12+2 = 24) + D (11) = 35

Ingresos: A (155+186+31 = 372) + D (209) = 581.0

Lácteos = producto C

Cantidad: 7; Ingresos: 280.0

Snacks = producto B

Cantidad: 5+3 = 8; Ingresos: 125+75 = 200.0

Comprobaciones rápidas (consistencia)

Suma de ingresos por categoría: 581.0 + 280.0 + 200.0 = 1061.0
Coincide con la suma por producto: A(372) + B(200) + C(280) + D(209) = 1061.0.

Suma de cantidades: 35 + 7 + 8 = 50
Coincide con cantidades por producto: A(24) + B(8) + C(7) + D(11) = 50.

En resumen: el join funcionó bien (cada venta quedó con su categoría) y la agregación por categoría refleja exactamente la suma de sus productos asociados.